# QSS 45 Final Project — Notebook 2: Preprocessing & Feature Engineering

**Pooled cross-section design (initial assignment).** This notebook pools two LinkedIn snapshots:

- **April 2024** — Kaggle "LinkedIn Job Postings 2024" supplement (~8,627 rows in our 5 industries)
- **May 2026** — direct scrape from `01_scraping.ipynb` (~5,243 rows)

Combined corpus: **~13,870 postings** across `pharma_chem`, `legal_services`, `farming_forestry`, `insurance`, `patent_ip`. The `source_platform` column is preserved (`linkedin` vs `linkedin_2024`) so the **final paper can pivot to a 2024-vs-2026 temporal comparison** — the eventual research question. For the initial data-analysis assignment, we treat the pool as a single cross-section.

**Inputs:** every CSV in `data/raw/` matching `postings_<industry>*.csv`

**Output:** `data/processed/postings_clean.csv`

In [ ]:
# --- Project-root path bootstrap (added by repo-reorg) ---
# Ensures cwd is the project root regardless of where this notebook was
# launched from (root, code/, JupyterLab tree, VS Code, etc.).
import os
from pathlib import Path
_here = Path.cwd().resolve()
while not (_here / 'data').exists() and _here.parent != _here:
    _here = _here.parent
os.chdir(_here)
PROJECT_ROOT = _here
# ---------------------------------------------------------

import os
import re
import html
import hashlib
import warnings
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

warnings.filterwarnings('ignore')
load_dotenv()

RAW_DIR   = Path('data/raw')
PROC_DIR  = Path('data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = int(os.getenv('RANDOM_SEED', 42))

print('Libraries loaded.')

## 1. Load Raw Data

In [16]:
INDUSTRY_KEYS = ['pharma_chem', 'legal_services', 'farming_forestry', 'insurance', 'patent_ip']

TIME_BINS_ORDERED = ['pre_gpt3', 'gpt3_era', 'copilot_chatgpt', 'gpt4_harvey', 'gpt4o_present']

AI_BREAKPOINTS = {
    'GPT-3':             '2020-10-01',
    'GitHub Copilot GA': '2022-06-01',
    'ChatGPT':           '2022-11-30',
    'GPT-4':             '2023-03-14',
    'Harvey AI launch':  '2023-11-01',
    'GPT-4o':            '2024-05-13',
    'Claude 3.5 Sonnet': '2024-06-20',
    'OpenAI o1':         '2024-09-12',
    'Claude 3.7 Sonnet': '2025-02-24',
    'OpenAI o3/o4-mini': '2025-04-16',
}

TIME_BIN_DATES = {
    'pre_gpt3':        ('2015-01-01', '2020-09-30'),
    'gpt3_era':        ('2020-10-01', '2022-05-31'),
    'copilot_chatgpt': ('2022-06-01', '2023-02-28'),
    'gpt4_harvey':     ('2023-03-01', '2024-04-30'),
    'gpt4o_present':   ('2024-05-01', '2030-12-31'),   # open-ended: covers current and future scrapes
}


def assign_time_bin(dt) -> str:
    """Assign a time-bin label to a pandas Timestamp."""
    if pd.isna(dt):
        return 'unknown'
    for label, (start, end) in TIME_BIN_DATES.items():
        if pd.Timestamp(start) <= dt <= pd.Timestamp(end):
            return label
    return 'unknown'


# ---------------------------------------------------------------------------
# LOADER — pooled cross-section design
# ---------------------------------------------------------------------------
# Pulls every CSV matching postings_<industry>*.csv, which includes:
#   - postings_<industry>.csv             — May-2026 scrape (this project)
#   - postings_<industry>_2024.csv        — Kaggle April-2024 supplement
#   - postings_<industry>_<other>.csv     — any future imports via 00_import_external_dataset.ipynb
#
# All sources are stacked into one DataFrame. The `source_platform` column
# (set by the scraper or by the import notebook) preserves provenance:
#   - 'linkedin'        → May 2026 scrape
#   - 'linkedin_2024'   → Kaggle April 2024
# Use this column downstream when you want period-stratified analyses.
# ---------------------------------------------------------------------------

frames = []
files_loaded = []
for key in INDUSTRY_KEYS:
    # Glob picks up both the base file and any period-suffixed variants
    matches = sorted(RAW_DIR.glob(f'postings_{key}*.csv'))
    if not matches:
        print(f'  {key}: NO FILES FOUND — skipping')
        continue
    for p in matches:
        df_i = pd.read_csv(p, dtype=str, low_memory=False)
        # Defensive: make sure industry_key is set (some imports may omit it)
        if 'industry_key' not in df_i.columns or df_i['industry_key'].isna().all():
            df_i['industry_key'] = key
        frames.append(df_i)
        files_loaded.append(p.name)
        print(f'  {p.name:50s}  {len(df_i):>6,} rows')

if not frames:
    raise FileNotFoundError('No raw posting CSVs found. Run 01_scraping.ipynb or 00_import_external_dataset.ipynb first.')

df = pd.concat(frames, ignore_index=True)

# Source-platform sanity check — if missing, fill with 'linkedin' (back-compat with old scrapes)
if 'source_platform' not in df.columns:
    df['source_platform'] = 'linkedin'
df['source_platform'] = df['source_platform'].fillna('linkedin')

print(f'\nPooled cross-section: {len(df):,} rows from {len(files_loaded)} file(s), {df.shape[1]} columns')
print('\nSource-platform breakdown (preserves provenance for downstream analyses):')
print(df['source_platform'].value_counts().to_string())
print('\nPer-industry counts:')
print(df['industry_key'].value_counts().to_string())
df.head(3)


  postings_pharma_chem.csv                               806 rows
  postings_pharma_chem_2024.csv                        4,221 rows
  postings_legal_services.csv                            924 rows
  postings_legal_services_2024.csv                     1,494 rows
  postings_farming_forestry.csv                        1,070 rows
  postings_farming_forestry_2024.csv                     135 rows
  postings_insurance.csv                               1,170 rows
  postings_insurance_2024.csv                          2,652 rows
  postings_patent_ip.csv                               1,273 rows
  postings_patent_ip_2024.csv                            125 rows

Pooled cross-section: 13,870 rows from 10 file(s), 14 columns

Source-platform breakdown (preserves provenance for downstream analyses):
source_platform
linkedin_2024    8627
linkedin         5243

Per-industry counts:
industry_key
pharma_chem         5027
insurance           3822
legal_services      2418
patent_ip           1398
farming

,job_id,title,company,industry_key,industry_label,seniority,location,date_posted,description,skills_tags,employment_type,source_platform,source_url,scraped_at
0,5c2c3a4112f2,Manufacturing Associate IV,Biogen,pharma_chem,Pharmaceutical / Chemical Manufacturing,NaN,"Triangle, NC (On-site)",NaN,About the job\nThis role operates on a 2 2 3 s...,NaN,NaN,linkedin,https://www.linkedin.com/jobs/view/4413529052/,2026-05-12T06:50:13.600799
1,c26a66a023aa,Critical Chemicals Analyst,Booz Allen Hamilton,pharma_chem,Pharmaceutical / Chemical Manufacturing,NaN,"Alexandria, VA (Remote)",2026-05-12,About the job\nJob Number: R0236134\n\nCritica...,NaN,NaN,linkedin,https://www.linkedin.com/jobs/view/4413293032/,2026-05-12T06:50:57.445299
2,f490e4aa7bab,Medical Practice Representative I - Surgery,University of Maryland Faculty Physicians,pharma_chem,Pharmaceutical / Chemical Manufacturing,NaN,"Baltimore, MD (On-site)",2026-05-12,About the job\nVarious receptionist and cleric...,NaN,NaN,linkedin,https://www.linkedin.com/jobs/view/4411849102/,2026-05-12T06:51:39.659690


## 2. Deduplication

In [17]:
n_before = len(df)

def content_hash(row) -> str:
    """Stable hash over (title, company, first 200 chars of description)."""
    key = f'{str(row.get("title", "")).lower()}|{str(row.get("company", "")).lower()}|{str(row.get("description", ""))[:200].lower()}'
    return hashlib.md5(key.encode()).hexdigest()


df['_hash'] = df.apply(content_hash, axis=1)
df = df.drop_duplicates(subset='_hash').reset_index(drop=True)

# Also drop exact-URL duplicates within each platform
df = df.drop_duplicates(subset=['source_url', 'source_platform']).reset_index(drop=True)

n_after = len(df)
print(f'Deduplication: {n_before:,} → {n_after:,} rows ({n_before - n_after:,} duplicates removed)')

Deduplication: 13,870 → 12,647 rows (1,223 duplicates removed)


## 3. HTML Stripping and Text Normalisation

In [18]:
# ----------------------------------------------------------------------------
# Boilerplate stripping
# ----------------------------------------------------------------------------
# Two layers:
#   (a) Generic job-posting boilerplate (EEO statements, "apply now", etc.)
#   (b) LinkedIn-specific UI chrome that the scraper accidentally captured
#       (sidebar metadata, Premium upsells, screen-reader overlays, language
#        selector footer). Validated to preserve every AI-term flag while
#        stripping ~13% of corpus volume as pure noise.
# ----------------------------------------------------------------------------

GENERIC_BOILERPLATE = [
    r'equal opportunity employer[^.]*\.?',
    r'we are an eoe[^.]*\.?',
    r'we offer a competitive salary[^.]*\.?',
    r'apply now[^.]*\.?',
    r'click (here )?to apply[^.]*\.?',
    r'(view|see) full job description[^.]*\.?',
    r'show (more|less)[^.]*\.?',
    r'job (type|id|ref(erence)?)\s*:?\s*[\w\-]+',
    r'(full[- ]time|part[- ]time|contract|permanent)\s*$',
]

LINKEDIN_BOILERPLATE = [
    # "Posted X time-units ago" — with or without "posted" prefix
    r'(posted\s+)?\d+\s+(day|week|month|hour|minute|min|sec|hr)s?\s+ago',
    r'posted\s+(yesterday|today|just\s+now)',
    # Sidebar engagement stats
    r'\b\d+\s+(applicants?|connections?\s+work\s+here|people\s+clicked\s+apply)',
    r'over\s+\d+\s+applicants',
    r'be\s+among\s+the\s+first\s+\d+\s+applicants',
    r'actively\s+reviewing\s+applicants',
    # Apply / share buttons
    r'easy\s+apply',
    r'see\s+more\s+jobs\s+like\s+this',
    r'similar\s+jobs',
    r'jobs?\s+you\s+might\s+be\s+interested\s+in',
    r'save\s+(this\s+)?job',
    r'share\s+(this\s+)?job',
    # Premium upsells
    r'linkedin\s+premium',
    r'try\s+premium[^.\n]*',
    r'unlock\s+premium[^.\n]*',
    r'get\s+premium',
    # Accessibility overlay
    r'you\s+are\s+on\s+the\s+messaging\s+overlay[^.\n]*',
    r'press\s+enter\s+to\s+(open|close)[^.\n]*',
    r'to\s+open\s+the\s+dialog',
    # Footer / language selector
    r'about\s+·\s+accessibility\s+·[^\n]*',
    r'linkedin\s+corporation\s+©[^\n]*',
    r'\b(bahasa|tagalog|chinese|english)\b\s*(indonesia|malaysia)?',
    # Location chrome
    r'\((on-site|hybrid|remote)\)',
    r'united\s+states\s+·\s+(remote|on-site|hybrid)',
    # Common in-description scrape artifacts
    r'^about\s+the\s+job\s*',
]

BOILERPLATE_PATTERNS = GENERIC_BOILERPLATE + LINKEDIN_BOILERPLATE
_BOILERPLATE_RE = re.compile('|'.join(f'(?:{p})' for p in BOILERPLATE_PATTERNS),
                              flags=re.IGNORECASE | re.MULTILINE)


def clean_text(raw: str) -> str:
    """
    Pipeline:
      1. Decode HTML entities
      2. Strip HTML tags via BeautifulSoup (handles Indeed/Wayback variants)
      3. Strip generic + LinkedIn-specific boilerplate
      4. Collapse whitespace

    Validated: preserves AI-term flag counts (flag_ai, flag_ml, flag_llm, ...)
    while removing ~13% of total chars (LinkedIn UI chrome).
    """
    if not isinstance(raw, str) or not raw.strip():
        return ''
    text = html.unescape(raw)
    if '<' in text and '>' in text:
        text = BeautifulSoup(text, 'lxml').get_text(separator=' ')
    text = _BOILERPLATE_RE.sub(' ', text)
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'  +', ' ', text)
    return text.strip()


df['description_clean'] = df['description'].apply(clean_text)
df['title_clean']       = df['title'].apply(clean_text)

print(f'Text cleaned. Sample description (first 200 chars):')
print(df['description_clean'].iloc[0][:200])


Text cleaned. Sample description (first 200 chars):
This role operates on a 2 2 3 schedule for Day Shift- 6p-6a. About This Role Our Manufacturing Associates perform processing steps and manufacturing support activities in our Drug Substance facility i


## 4. Filter Short Postings (< 50 Words)

In [19]:
df['word_count'] = df['description_clean'].apply(lambda t: len(t.split()) if isinstance(t, str) else 0)

n_before = len(df)
df = df[df['word_count'] >= 50].reset_index(drop=True)
print(f'Short-posting filter: {n_before:,} → {len(df):,} rows ({n_before - len(df):,} dropped)')

Short-posting filter: 12,647 → 12,487 rows (160 dropped)


## 5. Date Parsing and Time-Bin Assignment

In [20]:
df['date_posted'] = pd.to_datetime(df['date_posted'], errors='coerce')

# Drop postings with no recoverable date (they cannot be assigned to a bin)
n_no_date = df['date_posted'].isna().sum()
print(f'Postings with no parseable date: {n_no_date:,} — these will be retained with time_bin="unknown"')

df['time_bin'] = df['date_posted'].apply(assign_time_bin)

# Apply ordered categorical type for correct sort order in plots
bin_order = pd.CategoricalDtype(categories=TIME_BINS_ORDERED + ['unknown'], ordered=True)
df['time_bin_cat'] = df['time_bin'].astype(bin_order)

df['year']  = df['date_posted'].dt.year
df['month'] = df['date_posted'].dt.to_period('M').astype(str)

print('\nTime-bin distribution:')
print(df['time_bin'].value_counts().reindex(TIME_BINS_ORDERED + ['unknown'], fill_value=0))

Postings with no parseable date: 802 — these will be retained with time_bin="unknown"

Time-bin distribution:
time_bin
pre_gpt3              0
gpt3_era              0
copilot_chatgpt       0
gpt4_harvey        7721
gpt4o_present      3964
unknown             802
Name: count, dtype: int64


## 6. Feature Engineering

### 6a. AI-Term Binary Flags

In [21]:
# Term → compiled regex mapping
# Word-boundary anchors (\b) prevent partial matches (e.g., "AI" inside "BRAIN")
AI_TERMS: Dict[str, re.Pattern] = {
    'flag_machine_learning':  re.compile(r'\bmachine[- ]?learning\b', re.I),
    'flag_artificial_intel':  re.compile(r'\bartificial intelligence\b|\b\bAI\b', re.I),
    'flag_automation':        re.compile(r'\bautomati[onsed]+\b', re.I),
    'flag_llm':               re.compile(r'\bLLM[s]?\b|\blarge language model', re.I),
    'flag_generative_ai':     re.compile(r'\bgenerative AI\b|\bgen[- ]?AI\b', re.I),
    'flag_python':            re.compile(r'\bpython\b', re.I),
    'flag_prompt_eng':        re.compile(r'\bprompt engineering\b|\bprompt design\b', re.I),
    'flag_nlp':               re.compile(r'\bNLP\b|\bnatural language process', re.I),
    'flag_deep_learning':     re.compile(r'\bdeep[- ]?learning\b|\bneural network', re.I),
    'flag_data_science':      re.compile(r'\bdata science\b|\bdata scientist', re.I),
    'flag_gpt':               re.compile(r'\bGPT[- ]?[0-9]*\b|\bChatGPT\b|\bOpenAI\b', re.I),
    'flag_copilot':           re.compile(r'\bCopilot\b|\bGitHub Copilot\b', re.I),
    'flag_predictive':        re.compile(r'\bpredictive analytics\b|\bpredictive model', re.I),
    'flag_cloud_ml':          re.compile(r'\bSageMaker\b|\bAzure ML\b|\bVertex AI\b|\bBigQuery ML\b', re.I),
}


def apply_ai_flags(text: str) -> Dict[str, int]:
    """Return {flag_name: 0/1} for every AI term in the description."""
    if not isinstance(text, str):
        return {k: 0 for k in AI_TERMS}
    return {k: int(bool(pat.search(text))) for k, pat in AI_TERMS.items()}


flags_df = df['description_clean'].apply(lambda t: pd.Series(apply_ai_flags(t)))
df = pd.concat([df, flags_df], axis=1)

flag_cols = list(AI_TERMS.keys())
df['ai_term_count'] = df[flag_cols].sum(axis=1)
# DV threshold: >=1 (any AI term). Originally >=2 but produced a 4-6% positive
# class which is unlearnable from non-text features alone (F1=0.00 at threshold 0.5).
# >=1 raises the positive share to ~25-30%, well within reach of a tree model.
AI_EXPOSURE_THRESHOLD = 1
df['ai_exposure_binary'] = (df['ai_term_count'] >= AI_EXPOSURE_THRESHOLD).astype(int)

print(f'AI flags created: {flag_cols}')
print(f'Mean AI term count: {df["ai_term_count"].mean():.2f}')
print(f'High-AI-exposure share (>= {AI_EXPOSURE_THRESHOLD} AI term): {df["ai_exposure_binary"].mean():.1%}')

AI flags created: ['flag_machine_learning', 'flag_artificial_intel', 'flag_automation', 'flag_llm', 'flag_generative_ai', 'flag_python', 'flag_prompt_eng', 'flag_nlp', 'flag_deep_learning', 'flag_data_science', 'flag_gpt', 'flag_copilot', 'flag_predictive', 'flag_cloud_ml']
Mean AI term count: 0.23
High-AI-exposure share (>= 1 AI term): 16.3%


### 6b. Seniority Level — Ordinal Encoding

In [23]:
# Map heterogeneous seniority strings (from LinkedIn, Indeed, Wayback) to a single scale
SENIORITY_MAP = {
    # LinkedIn labels
    'internship':    0,
    'entry level':   1,
    'associate':     2,
    'mid-senior level': 3,
    'director':      4,
    'executive':     5,
    # Common free-text equivalents
    'junior':        1,
    'senior':        3,
    'lead':          3,
    'staff':         3,
    'principal':     4,
    'vp':            4,
    'vice president': 4,
    'manager':       3,
    'c-suite':       5,
}


def parse_seniority(raw: str) -> Optional[int]:
    """Convert raw seniority string to ordinal 0–5; return NaN if unresolvable."""
    if not isinstance(raw, str) or not raw.strip():
        return np.nan
    key = raw.lower().strip()
    # Direct match
    if key in SENIORITY_MAP:
        return SENIORITY_MAP[key]
    # Fuzzy: check if any canonical label appears as a substring
    for label, val in sorted(SENIORITY_MAP.items(), key=lambda x: -len(x[0])):
        if label in key:
            return val
    # Fallback: infer from title
    return np.nan


def infer_seniority_from_title(title: str) -> Optional[int]:
    """Infer seniority from job title when seniority field is missing."""
    if not isinstance(title, str):
        return np.nan
    t = title.lower()
    for label, val in SENIORITY_MAP.items():
        if label in t:
            return val
    return 2  # default: mid-level (most common in job-board postings)


df['seniority_ordinal'] = df['seniority'].apply(parse_seniority)

# Fill missing from title, then default to 2
mask = df['seniority_ordinal'].isna()
df.loc[mask, 'seniority_ordinal'] = df.loc[mask, 'title_clean'].apply(infer_seniority_from_title)
df['seniority_ordinal'] = df['seniority_ordinal'].fillna(2).astype(int)

print('Seniority ordinal distribution:')
print(df['seniority_ordinal'].value_counts().sort_index())

Seniority ordinal distribution:
seniority_ordinal
0      91
1    1418
2    6109
3    3974
4     686
5     209
Name: count, dtype: int64


### 6c. Industry — Categorical Code

In [24]:
INDUSTRY_CODE_MAP = {
    'pharma_chem':     0,
    'legal_services':  1,
    'farming_forestry': 2,
    'insurance':       3,
    'patent_ip':       4,
}

df['industry_code'] = df['industry_key'].map(INDUSTRY_CODE_MAP).fillna(-1).astype(int)
df['industry_cat']  = pd.Categorical(df['industry_key'], categories=list(INDUSTRY_CODE_MAP.keys()))

print('Industry distribution:')
print(df['industry_key'].value_counts())

Industry distribution:
industry_key
pharma_chem         4581
insurance           3360
legal_services      2215
patent_ip           1231
farming_forestry    1100
Name: count, dtype: int64


### 6d. Employment Type — Normalisation

In [25]:
EMP_TYPE_MAP = {
    r'full[- ]?time':  'Full-time',
    r'part[- ]?time':  'Part-time',
    r'contract':       'Contract',
    r'freelance':      'Contract',
    r'temporary':      'Temporary',
    r'intern':         'Internship',
    r'volunteer':      'Volunteer',
}


def normalise_emp_type(raw: str) -> str:
    """Map raw employment_type string to a canonical label."""
    if not isinstance(raw, str):
        return 'Unknown'
    for pat, label in EMP_TYPE_MAP.items():
        if re.search(pat, raw, re.I):
            return label
    return 'Unknown'


df['employment_type_clean'] = df['employment_type'].apply(normalise_emp_type)

print('Employment type distribution:')
print(df['employment_type_clean'].value_counts())

Employment type distribution:
employment_type_clean
Full-time     6666
Unknown       4774
Contract       822
Part-time      127
Temporary       54
Internship      42
Volunteer        2
Name: count, dtype: int64


### 6e. AI-Term Frequency (Continuous DV)

In [26]:
df['ai_term_freq'] = df['ai_term_count'] / df['word_count'].clip(lower=1)

print('AI term frequency statistics:')
print(df['ai_term_freq'].describe())

AI term frequency statistics:
count    12487.000000
mean         0.000444
std          0.001891
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.070000
Name: ai_term_freq, dtype: float64


### 6f. BERT Sentence Embeddings (Optional — for clustering / similarity)

In [27]:
COMPUTE_EMBEDDINGS = False  # Set True to compute BERT embeddings

if COMPUTE_EMBEDDINGS:
    from sentence_transformers import SentenceTransformer
    import torch

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Computing embeddings on: {device}')

    model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

    # Encode the first 512 characters of each description (MiniLM is context-limited)
    texts = df['description_clean'].fillna('').apply(lambda t: t[:512]).tolist()
    embeddings = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

    emb_df = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
    emb_df.to_parquet(PROC_DIR / 'embeddings.parquet', index=False)
    print(f'Embeddings saved: shape {embeddings.shape}')
else:
    print('Skipping BERT embeddings (COMPUTE_EMBEDDINGS=False).')

Skipping BERT embeddings (COMPUTE_EMBEDDINGS=False).


## 7. Final Column Selection and Save

In [28]:
KEEP_COLS = [
    # Identifiers
    'job_id', 'title_clean', 'company', 'source_platform', 'source_url',
    # Industry / temporal
    'industry_key', 'industry_label', 'industry_code', 'industry_cat',
    'date_posted', 'year', 'month',
    'time_bin', 'time_bin_cat',
    # Job metadata
    'location', 'seniority', 'seniority_ordinal',
    'employment_type_clean',
    'skills_tags',
    # Text + length
    'description_clean', 'word_count',
    # AI flags & DVs
    *list(AI_TERMS.keys()),
    'ai_term_count', 'ai_term_freq', 'ai_exposure_binary',
]

# Keep only columns that exist (guards against partial scraping)
keep = [c for c in KEEP_COLS if c in df.columns]
df_clean = df[keep].copy()

out_path = PROC_DIR / 'postings_clean.csv'
df_clean.to_csv(out_path, index=False)

print(f'Clean dataset saved: {out_path}')
print(f'Shape: {df_clean.shape}')
df_clean.dtypes

Clean dataset saved: data/processed/postings_clean.csv
Shape: (12487, 38)


job_id                              str
title_clean                         str
company                             str
source_platform                     str
source_url                          str
industry_key                        str
industry_label                      str
industry_code                     int64
industry_cat                   category
date_posted              datetime64[us]
year                            float64
month                               str
time_bin                            str
time_bin_cat                   category
location                            str
seniority                           str
seniority_ordinal                 int64
employment_type_clean               str
skills_tags                         str
description_clean                   str
word_count                        int64
flag_machine_learning             int64
flag_artificial_intel             int64
flag_automation                   int64
flag_llm                          int64


## 8. Preprocessing Summary

In [29]:
print('=' * 60)
print('PREPROCESSING SUMMARY')
print('=' * 60)
print(f'Total postings (clean):  {len(df_clean):,}')
print(f'Industries:              {df_clean["industry_key"].nunique()}')
print(f'Date range:              {df_clean["date_posted"].min().date()} – {df_clean["date_posted"].max().date()}')
print(f'Time bins covered:       {df_clean["time_bin"].nunique()}')
print(f'Missing dates:           {df_clean["date_posted"].isna().sum()}')
print(f'Median posting length:   {df_clean["word_count"].median():.0f} words')
print(f'Mean AI term count:      {df_clean["ai_term_count"].mean():.2f}')
print(f'High-AI-exposure share:  {df_clean["ai_exposure_binary"].mean():.1%}')
print()
print('Postings per industry:')
print(df_clean['industry_key'].value_counts().to_string())
print()
print('Postings per time bin:')
print(df_clean['time_bin'].value_counts().reindex(TIME_BINS_ORDERED, fill_value=0).to_string())

PREPROCESSING SUMMARY
Total postings (clean):  12,487
Industries:              5
Date range:              2024-04-05 – 2026-05-18
Time bins covered:       3
Missing dates:           802
Median posting length:   606 words
Mean AI term count:      0.23
High-AI-exposure share:  16.3%

Postings per industry:
industry_key
pharma_chem         4581
insurance           3360
legal_services      2215
patent_ip           1231
farming_forestry    1100

Postings per time bin:
time_bin
pre_gpt3              0
gpt3_era              0
copilot_chatgpt       0
gpt4_harvey        7721
gpt4o_present      3964
